# CosmicWatch: Breakthrough Listen Data Exploration & Anomaly Triage

**Flagship Use Case**: Radio SETI & Astronomical Candidate Prioritization.

> **SCIENTIFIC DISCLAIMER**: CosmicWatch is an anomaly detection system that identifies unusual radio spectral patterns. It is **NOT** an alien detector and does not establish extraterrestrial origin.

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from ml.data.load_breakthrough import BreakthroughListenLoader
from ml.data.synthetic_generator import SyntheticSignalGenerator
from ml.preprocessing.normalize import preprocess_waterfall
from ml.inference.classifier import SignalDetector
from ml.anomaly_detection.anomaly_detector import AnomalyDetector, AstronomicalFeatureExtractor
from backend.services.candidate_engine import CandidateEngine

## 1. Load Breakthrough Listen Sample with BLIMPY

In [ ]:
sample_file = '../data/sample_bl/voyager1_sample.h5'
loader = BreakthroughListenLoader(sample_file)
window = loader.extract_window()

print(f'Target: {window.target_name}')
print(f'Telescope: {window.telescope}')
print(f'Data matrix shape: {window.data.shape}')
print(f'Frequency range: {window.freq_start_mhz:.4f} - {window.freq_end_mhz:.4f} MHz')

## 2. Visualize Spectrogram Waterfall

In [ ]:
plt.figure(figsize=(10, 4), dpi=120)
plt.imshow(window.data, aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(label='Normalized Power')
plt.title(f'Breakthrough Listen Spectrogram: {window.target_name}')
plt.xlabel('Frequency Channel')
plt.ylabel('Time Integration')
plt.show()

## 3. Run Dual-Path AI Anomaly Detection & Scoring

In [ ]:
detector = SignalDetector('../ml/models_saved/signal_classifier.pt')
anom_detector = AnomalyDetector('../ml/models_saved/anomaly_detector.joblib')
engine = CandidateEngine('../ml/config/scoring_weights.yaml')

sig_res = detector.predict(window)
anom_res = anom_detector.score(window, latent_embedding=sig_res['embedding'])

metrics = anom_res['astronomical_metrics']
eval_res = engine.evaluate_candidate(
    signal_confidence=sig_res['signal_confidence'],
    anomaly_score=anom_res['anomaly_score'],
    persistence=metrics['persistence'],
    narrowband_likelihood=metrics['narrowband_likelihood'],
    rfi_likelihood=0.15,
    on_source_detected=True,
    off_source_detected=False,
    frequency_drift_detected=False
)

print(f'CosmicWatch Score: {eval_res["candidate_score"]}/100')
print(f'Status:            {eval_res["status"]}')
print(f'Recommendation:    {eval_res["recommendation"]}')